# Mitra Classifier — Use an Exported Predictor

**Notebook Specification:** DIMER Notebook Specification v1.0  
**Profile:** `ARTIFACT-INFERENCE`  
**Release status:** Candidate — static conformance checks are automated; clean Google Colab execution of this exact revision remains the release gate.

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/mitra-classifier-pipeline)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/mitra-classifier-pipeline/blob/main/tutorials/mitra_classifier_predictor_inference_colab.ipynb)
[![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-autogluon%2Fmitra--classifier-ffcc4d?style=flat)](https://huggingface.co/autogluon/mitra-classifier)
[![Upstream](https://img.shields.io/badge/Upstream-autogluon%2Fautogluon-181717?style=flat&logo=github&logoColor=white)](https://github.com/autogluon/autogluon)
[![arXiv](https://img.shields.io/badge/arXiv-2510.21204-b31b1b.svg)](https://arxiv.org/abs/2510.21204)

Someone ran the main tutorial and handed you `mitra-predictor.zip`. You want predictions for new rows, and you want to know what you are loading before you load it: an AutoGluon predictor directory is Python objects (pickle) plus the Mitra weights and the training context, and loading it executes code.

**You do not need the original DIMER ZIP, `model.safetensors`, `config.json`, or DIMER Workbench.** The ZIP already contains everything the predictor needs.

**This notebook is inference-only:** it does not train, fine-tune, refit preprocessing, reacquire the base checkpoint, or validate production fitness.

Reference: [MODEL_CARD.md](https://github.com/kurtvalcorza/mitra-classifier-pipeline/blob/main/MODEL_CARD.md).

**By the end of this notebook you will be able to:**
- **Check** a predictor archive before loading it: expected digest, safe paths, exactly one predictor root.
- **Read** the bundle's provenance and confirm the runtime matches the one it was built with.
- **Score** a new CSV whose columns may be reordered or superset, and get a `predictions.csv` with class probabilities.

## Prerequisites
- The `mitra-predictor.zip` from the main tutorial, and ideally the SHA-256 it printed at export.
- A CSV of new rows with the predictor's feature columns (the target column is not needed).
- Google Colab with **Python 3.12**; inference runs on CPU. Completion time varies with dependency-cache state, archive size, and hardware, so no fixed runtime is promised.

`mitra-predictor.zip` → verify → reload predictor → upload new CSV → predict → download `predictions.csv`

**Data handling:** the uploaded predictor ZIP and inference CSV are processed inside the Google Colab runtime. This notebook does not send inference rows to DIMER, Hugging Face, or another model API. Google Colab remains the hosting environment, so its data-handling policies still apply.


## 1. Install the matching runtime

The exported predictor was created with AutoGluon 1.5.0 and can only be trusted to load under the same version, so this notebook pins the same Mitra-capable runtime as the tutorial. Pip may replace Colab's preinstalled PyTorch to satisfy it; if PyTorch was already imported and the installed version changes, the cell asks you to restart the session and rerun from the top.

The dependency graph is installed from an exact-version lock generated into `tutorials/requirements-inference.lock.txt`.

The lock is rooted in the exact Mitra-capable requirement `autogluon.tabular[mitra]==1.5.0`; the committed lock then pins its resolved dependency graph.


In [ ]:
import importlib.metadata as importlib_metadata
import sys

if sys.version_info[:2] != (3, 12):
    raise RuntimeError(
        f'This notebook release lock targets Python 3.12, but this runtime is {sys.version.split()[0]}. '
        'Use a supported Google Colab Python 3.12 runtime, restart the session, and run top-to-bottom.'
    )

PREINSTALL_TORCH_VERSION = importlib_metadata.version('torch')
TORCH_WAS_IMPORTED = 'torch' in sys.modules
print('PyTorch before install:', PREINSTALL_TORCH_VERSION)

from pathlib import Path
LOCKED_REQUIREMENTS = '#\n# This file is autogenerated by pip-compile with Python 3.12\n# by the following command:\n#\n#    pip-compile --output-file=tutorials/requirements-inference.lock.txt --strip-extras tutorials/requirements-inference.in\n#\nantlr4-python3-runtime==4.9.3\n    # via omegaconf\nautogluon-common==1.5.0\n    # via\n    #   autogluon-core\n    #   autogluon-features\nautogluon-core==1.5.0\n    # via autogluon-tabular\nautogluon-features==1.5.0\n    # via autogluon-tabular\nautogluon-tabular==1.5.0\n    # via\n    #   -r tutorials/requirements-inference.in\n    #   autogluon-tabular\nboto3==1.43.91\n    # via\n    #   autogluon-common\n    #   autogluon-core\nbotocore==1.43.91\n    # via\n    #   boto3\n    #   s3transfer\ncertifi==2026.7.22\n    # via requests\ncharset-normalizer==3.5.1\n    # via requests\ncloudpickle==3.1.2\n    # via joblib\ncontourpy==1.3.3\n    # via matplotlib\ncycler==0.12.1\n    # via matplotlib\neinops==0.8.2\n    # via autogluon-tabular\neinx==0.4.3\n    # via autogluon-tabular\nfilelock==3.32.6\n    # via\n    #   huggingface-hub\n    #   torch\n    #   transformers\nfonttools==4.64.0\n    # via matplotlib\nfrozendict==2.4.7\n    # via einx\nfsspec==2026.7.0\n    # via\n    #   huggingface-hub\n    #   torch\nhf-xet==1.6.0\n    # via huggingface-hub\nhuggingface-hub==0.36.2\n    # via\n    #   autogluon-tabular\n    #   huggingface-hub\n    #   tokenizers\n    #   transformers\nidna==3.19\n    # via requests\njinja2==3.1.6\n    # via torch\njmespath==1.1.0\n    # via\n    #   boto3\n    #   botocore\njoblib==1.6.0\n    # via\n    #   autogluon-common\n    #   scikit-learn\nkiwisolver==1.5.1\n    # via matplotlib\nloguru==0.7.3\n    # via autogluon-tabular\nmarkupsafe==3.0.3\n    # via jinja2\nmatplotlib==3.10.9\n    # via autogluon-core\nmpmath==1.3.0\n    # via sympy\nnetworkx==3.6.1\n    # via\n    #   autogluon-core\n    #   autogluon-tabular\n    #   torch\nnumpy==2.3.5\n    # via\n    #   autogluon-common\n    #   autogluon-core\n    #   autogluon-features\n    #   autogluon-tabular\n    #   contourpy\n    #   einx\n    #   matplotlib\n    #   pandas\n    #   safetensors\n    #   scikit-learn\n    #   scipy\n    #   transformers\nnvidia-cublas-cu12==12.8.4.1\n    # via\n    #   nvidia-cudnn-cu12\n    #   nvidia-cusolver-cu12\n    #   torch\nnvidia-cuda-cupti-cu12==12.8.90\n    # via torch\nnvidia-cuda-nvrtc-cu12==12.8.93\n    # via torch\nnvidia-cuda-runtime-cu12==12.8.90\n    # via torch\nnvidia-cudnn-cu12==9.10.2.21\n    # via torch\nnvidia-cufft-cu12==11.3.3.83\n    # via torch\nnvidia-cufile-cu12==1.13.1.3\n    # via torch\nnvidia-curand-cu12==10.3.9.90\n    # via torch\nnvidia-cusolver-cu12==11.7.3.90\n    # via torch\nnvidia-cusparse-cu12==12.5.8.93\n    # via\n    #   nvidia-cusolver-cu12\n    #   torch\nnvidia-cusparselt-cu12==0.7.1\n    # via torch\nnvidia-nccl-cu12==2.27.5\n    # via torch\nnvidia-nvjitlink-cu12==12.8.93\n    # via\n    #   nvidia-cufft-cu12\n    #   nvidia-cusolver-cu12\n    #   nvidia-cusparse-cu12\n    #   torch\nnvidia-nvshmem-cu12==3.3.20\n    # via torch\nnvidia-nvtx-cu12==12.8.90\n    # via torch\nomegaconf==2.3.1\n    # via autogluon-tabular\npackaging==26.3\n    # via\n    #   huggingface-hub\n    #   matplotlib\n    #   transformers\npandas==2.3.3\n    # via\n    #   autogluon-common\n    #   autogluon-core\n    #   autogluon-features\n    #   autogluon-tabular\npillow==12.3.0\n    # via matplotlib\npsutil==7.1.3\n    # via autogluon-common\npyarrow==20.0.0\n    # via autogluon-common\npyparsing==3.3.2\n    # via matplotlib\npython-dateutil==2.9.0.post0\n    # via\n    #   botocore\n    #   matplotlib\n    #   pandas\npytz==2026.3.post1\n    # via pandas\npyyaml==6.0.3\n    # via\n    #   autogluon-common\n    #   huggingface-hub\n    #   omegaconf\n    #   transformers\nregex==2026.9.10\n    # via transformers\nrequests==2.34.2\n    # via\n    #   autogluon-common\n    #   autogluon-core\n    #   huggingface-hub\n    #   transformers\ns3transfer==0.19.2\n    # via boto3\nsafetensors==0.8.0\n    # via\n    #   huggingface-hub\n    #   transformers\nscikit-learn==1.7.2\n    # via\n    #   autogluon-core\n    #   autogluon-features\n    #   autogluon-tabular\nscipy==1.16.3\n    # via\n    #   autogluon-core\n    #   autogluon-tabular\n    #   scikit-learn\nsix==1.17.0\n    # via python-dateutil\nsympy==1.14.0\n    # via\n    #   einx\n    #   torch\nthreadpoolctl==3.6.0\n    # via scikit-learn\ntokenizers==0.22.2\n    # via transformers\ntorch==2.9.1\n    # via\n    #   autogluon-tabular\n    #   huggingface-hub\n    #   safetensors\ntqdm==4.70.0\n    # via\n    #   autogluon-common\n    #   autogluon-core\n    #   huggingface-hub\n    #   transformers\ntransformers==4.57.6\n    # via autogluon-tabular\ntriton==3.5.1\n    # via torch\ntyping-extensions==4.16.0\n    # via\n    #   huggingface-hub\n    #   torch\ntzdata==2026.3\n    # via pandas\nurllib3==2.7.0\n    # via\n    #   botocore\n    #   requests\n\n# The following packages are considered to be unsafe in a requirements file:\n# setuptools\n'
Path('/content/mitra-inference-requirements.lock.txt').write_text(LOCKED_REQUIREMENTS, encoding='utf-8')
%pip install -q -r /content/mitra-inference-requirements.lock.txt

INSTALLED_TORCH_VERSION = importlib_metadata.version('torch')
AUTOGLUON_VERSION = importlib_metadata.version('autogluon.tabular')
if TORCH_WAS_IMPORTED and INSTALLED_TORCH_VERSION != PREINSTALL_TORCH_VERSION:
    raise RuntimeError('pip changed PyTorch after it had already been imported. Use Runtime → Restart session, then run the notebook top-to-bottom.')

import torch

print('Python:', sys.version.split()[0])
print('AutoGluon:', AUTOGLUON_VERSION)
print('PyTorch:', torch.__version__)
print('PyTorch CUDA build:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())


## 2. Upload and validate `mitra-predictor.zip`

Upload exactly one predictor ZIP exported by the main tutorial. Before anything is loaded, the notebook:

- computes the ZIP's SHA-256 and compares it with `EXPECTED_ZIP_SHA256` if you pasted one;
- rejects absolute paths, `..` traversal, backslash-style ambiguous member paths, symlinks, members escaping the extraction root, and archives whose declared expanded size exceeds 4 GiB;
- extracts into a fresh directory and requires exactly one AutoGluon predictor root (`predictor.pkl`).

**Trust boundary:** `TabularPredictor.load(...)` deserializes Python model objects. Path-safe extraction stops an archive from writing outside its folder; it does **not** make an untrusted predictor safe to deserialize. Load only a ZIP you exported yourself or received from a trusted source, and when you have the digest printed at export, paste it in so substitution is caught here.

The archive must contain `tutorial_run_metadata.json` declaring artifact format `dimer-mitra-autogluon-predictor`, format version `1.0`, immutable base-model revision, runtime version, and required feature schema. Missing or malformed provenance is fatal before any predictor object is deserialized.

`artifact-manifest.json` is verified before deserialization: every listed file must exist with the recorded size and SHA-256, and unexpected unlisted files are rejected. This proves **internal archive consistency**, not sender authenticity. An attacker who can replace both the ZIP and its manifest can make them agree; use `EXPECTED_ZIP_SHA256` from a trusted channel when sender/archive authenticity matters.


In [ ]:
import hashlib
import json
import os
import shutil
import stat
import zipfile
from pathlib import Path

from google.colab import files

EXPECTED_ZIP_SHA256 = ''  # @param {type:'string'}
MAX_EXPANDED_BYTES = 4 * 1024 ** 3
EXTRACT_ROOT = Path('/content/mitra-predictor-upload')
if EXTRACT_ROOT.exists():
    shutil.rmtree(EXTRACT_ROOT)
EXTRACT_ROOT.mkdir(parents=True)

uploaded = files.upload()
zips = [(name, payload) for name, payload in uploaded.items() if name.lower().endswith('.zip')]
if len(zips) != 1:
    raise RuntimeError('Upload exactly one mitra-predictor.zip file.')

zip_name, zip_payload = zips[0]
ZIP_PATH = Path('/content') / Path(zip_name).name
ZIP_PATH.write_bytes(zip_payload)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

def safe_extract_zip(zip_path, destination):
    destination = destination.resolve()
    with zipfile.ZipFile(zip_path) as z:
        expanded_bytes = 0
        for info in z.infolist():
            if '\\' in info.filename:
                raise RuntimeError(f'Backslash archive member paths are not allowed: {info.filename!r}')
            member = Path(info.filename)
            if member.is_absolute() or '..' in member.parts:
                raise RuntimeError(f'Unsafe archive member path: {info.filename!r}')
            mode = (info.external_attr >> 16) & 0o170000
            if mode == stat.S_IFLNK:
                raise RuntimeError(f'Symlink entries are not allowed: {info.filename!r}')
            expanded_bytes += int(info.file_size)
            if expanded_bytes > MAX_EXPANDED_BYTES:
                raise RuntimeError(
                    f'Archive expanded size exceeds the {MAX_EXPANDED_BYTES / (1024 ** 3):.0f} GiB safety limit.'
                )
            target = (destination / member).resolve()
            if target != destination and destination not in target.parents:
                raise RuntimeError(f'Archive member escapes extraction root: {info.filename!r}')
        z.extractall(destination)

zip_digest = sha256_file(ZIP_PATH)
expected_zip_digest = EXPECTED_ZIP_SHA256.strip().lower()
if expected_zip_digest:
    if len(expected_zip_digest) != 64 or any(ch not in '0123456789abcdef' for ch in expected_zip_digest):
        raise ValueError('EXPECTED_ZIP_SHA256 must be a 64-character hexadecimal SHA-256 digest.')
    if zip_digest != expected_zip_digest:
        raise RuntimeError(f'Predictor ZIP checksum mismatch. Expected {expected_zip_digest}; got {zip_digest}.')
    print('✓ Predictor ZIP SHA-256 matches the expected digest.')
else:
    print('⚠ No expected predictor ZIP digest supplied; continue only if this archive came from a trusted source.')
print(f'ZIP: {ZIP_PATH.name}')
print(f'Size: {ZIP_PATH.stat().st_size / (1024 ** 2):.1f} MiB')
print('SHA-256:', zip_digest)

safe_extract_zip(ZIP_PATH, EXTRACT_ROOT)

candidates = sorted({p.parent for p in EXTRACT_ROOT.rglob('predictor.pkl')})
if len(candidates) != 1:
    raise RuntimeError(
        f'Expected exactly one AutoGluon predictor root containing predictor.pkl; found {len(candidates)}: {candidates}'
    )

PREDICTOR_ROOT = candidates[0].resolve()
print('✓ Predictor root:', PREDICTOR_ROOT)

all_extracted_files = [p.resolve() for p in EXTRACT_ROOT.rglob('*') if p.is_file()]
outside_predictor_root = [
    p for p in all_extracted_files
    if p != PREDICTOR_ROOT and PREDICTOR_ROOT not in p.parents
]
if outside_predictor_root:
    raise RuntimeError(
        'Predictor ZIP contains file(s) outside the single predictor root: '
        f'{[str(p.relative_to(EXTRACT_ROOT.resolve())) for p in outside_predictor_root]}'
    )

MANIFEST_PATH = PREDICTOR_ROOT / 'artifact-manifest.json'
if not MANIFEST_PATH.exists():
    raise RuntimeError('Required artifact-manifest.json is missing; refusing to deserialize an unverifiable predictor bundle.')
artifact_manifest = json.loads(MANIFEST_PATH.read_text())
if artifact_manifest.get('artifact_format') != 'dimer-mitra-autogluon-predictor' or artifact_manifest.get('artifact_format_version') != '1.0':
    raise RuntimeError(
        f"Unsupported artifact manifest format/version: {artifact_manifest.get('artifact_format')!r} / {artifact_manifest.get('artifact_format_version')!r}."
    )
manifest_feature_schema_sha256 = str(artifact_manifest.get('feature_schema_sha256', '')).lower()
if len(manifest_feature_schema_sha256) != 64 or any(ch not in '0123456789abcdef' for ch in manifest_feature_schema_sha256):
    raise RuntimeError('Artifact manifest is missing a valid feature_schema_sha256.')
entries = artifact_manifest.get('files')
if not isinstance(entries, list) or not entries:
    raise RuntimeError('artifact-manifest.json must contain a non-empty files list.')
listed = {}
for entry in entries:
    if not isinstance(entry, dict) or not {'path', 'size_bytes', 'sha256'} <= set(entry):
        raise RuntimeError(f'Malformed artifact manifest entry: {entry!r}')
    rel = str(entry['path'])
    if '\\' in rel:
        raise RuntimeError(f'Backslash manifest paths are not allowed: {rel!r}')
    rel_path = Path(rel)
    if rel_path.is_absolute() or '..' in rel_path.parts or rel in listed:
        raise RuntimeError(f'Unsafe or duplicate artifact manifest path: {rel!r}')
    if rel == 'artifact-manifest.json':
        raise RuntimeError('artifact-manifest.json must not list itself.')
    listed[rel] = entry
actual = {
    p.relative_to(PREDICTOR_ROOT).as_posix()
    for p in PREDICTOR_ROOT.rglob('*')
    if p.is_file() and p.resolve() != MANIFEST_PATH.resolve()
}
expected = set(listed)
if actual != expected:
    raise RuntimeError(
        f'Artifact manifest file set mismatch. Missing={sorted(expected - actual)}; unexpected={sorted(actual - expected)}'
    )
for rel, entry in listed.items():
    p = PREDICTOR_ROOT / rel
    if p.stat().st_size != int(entry['size_bytes']):
        raise RuntimeError(f'Artifact manifest size mismatch for {rel!r}.')
    digest = sha256_file(p)
    expected_digest = str(entry['sha256']).lower()
    if len(expected_digest) != 64 or any(ch not in '0123456789abcdef' for ch in expected_digest):
        raise RuntimeError(f'Artifact manifest has invalid SHA-256 for {rel!r}.')
    if digest != expected_digest:
        raise RuntimeError(f'Artifact manifest SHA-256 mismatch for {rel!r}.')
print(f'✓ Artifact manifest verified: {len(listed)} files, exact file set, sizes, and SHA-256 digests.')


## 3. Load the predictor and inspect provenance

`tutorial_run_metadata.json`, written by the export step, records how the predictor was produced: model identity and digests, runtime versions, feature columns, data source, evaluation metrics, and whether fine-tuning was requested. Required provenance is validated before loading. If the artifact format/version, immutable model identity, feature schema, or recorded AutoGluon version is missing or incompatible, the notebook stops before deserialization.

The predictor itself is loaded with `TabularPredictor.load(...)`; the individual `.pkl` and `.pt` files inside the archive are implementation details and should not be opened by hand.

**What to look for:** `Problem type` of `binary` or `multiclass`, the `Models` line naming Mitra, and the list of required feature columns your CSV must contain.

Network fallback is explicitly disabled before `TabularPredictor.load(...)`; this notebook must reconstruct solely from the supplied artifact.

The ordered feature-name list is structurally validated and SHA-256-bound to both provenance and the artifact manifest before deserialization; after loading, the notebook cross-checks that list against AutoGluon's own input feature metadata.


In [ ]:
import json
from pathlib import Path

import pandas as pd
from autogluon.tabular import TabularPredictor

SUPPORTED_MODEL_ID = 'autogluon/mitra-classifier'
SUPPORTED_MODEL_REVISION = 'c425e9fa0910a6be1c494321792e7ba2a1367b1a'
SUPPORTED_WEIGHTS_SHA256 = 'e06a055e91a3baeffc37f9cf634d9e69a27d904b6686131dc3b702f9c0126b19'
SUPPORTED_CONFIG_SHA256 = '2c96c24dd25f64e92753f6f2ba00cc7833b9923459403dcd8504e8700c0995df'

METADATA_PATH = PREDICTOR_ROOT / 'tutorial_run_metadata.json'
if not METADATA_PATH.exists():
    raise RuntimeError('Required tutorial_run_metadata.json is missing; refusing to deserialize an artifact with unknown provenance.')
run_metadata = json.loads(METADATA_PATH.read_text())
required_provenance = [
    'artifact_format',
    'artifact_format_version',
    'base_model',
    'base_model_revision',
    'autogluon_version',
    'features',
    'weights_sha256',
    'config_sha256',
    'feature_schema_sha256',
]
missing_provenance = [k for k in required_provenance if not run_metadata.get(k)]
if missing_provenance:
    raise RuntimeError(f'Predictor provenance is incomplete; missing required field(s): {missing_provenance}')
raw_features = run_metadata['features']
if (
    not isinstance(raw_features, list)
    or not raw_features
    or len(raw_features) > 500
    or any(not isinstance(name, str) or not name.strip() for name in raw_features)
    or len(set(raw_features)) != len(raw_features)
):
    raise RuntimeError('Predictor provenance has an invalid required-feature schema.')
feature_schema_json = json.dumps(raw_features, ensure_ascii=False, separators=(',', ':'))
feature_schema_sha256 = hashlib.sha256(feature_schema_json.encode('utf-8')).hexdigest()
if run_metadata['feature_schema_sha256'] != feature_schema_sha256:
    raise RuntimeError('Predictor provenance feature-schema digest does not match the declared feature list.')
if manifest_feature_schema_sha256 != feature_schema_sha256:
    raise RuntimeError('Artifact manifest and predictor provenance disagree on the feature schema.')
if run_metadata['artifact_format'] != 'dimer-mitra-autogluon-predictor' or run_metadata['artifact_format_version'] != '1.0':
    raise RuntimeError(
        f"Unsupported predictor artifact {run_metadata['artifact_format']!r} version {run_metadata['artifact_format_version']!r}."
    )
if run_metadata['base_model'] != SUPPORTED_MODEL_ID or run_metadata['base_model_revision'] != SUPPORTED_MODEL_REVISION:
    raise RuntimeError(
        f"Unsupported model identity/revision: {run_metadata['base_model']!r} @ {run_metadata['base_model_revision']!r}."
    )
if run_metadata.get('weights_sha256') != SUPPORTED_WEIGHTS_SHA256 or run_metadata.get('config_sha256') != SUPPORTED_CONFIG_SHA256:
    raise RuntimeError('Predictor provenance weight/config digests do not match this notebook release.')
if artifact_manifest.get('base_model') != SUPPORTED_MODEL_ID or artifact_manifest.get('base_model_revision') != SUPPORTED_MODEL_REVISION:
    raise RuntimeError('Artifact manifest model identity/revision does not match this notebook release.')
recorded_ag = run_metadata['autogluon_version']
if recorded_ag != AUTOGLUON_VERSION:
    raise RuntimeError(
        f'Predictor was exported with AutoGluon {recorded_ag}, but this runtime has {AUTOGLUON_VERSION}. '
        'Use the recorded version for compatibility.'
    )

os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_DATASETS_OFFLINE'] = '1'
print('✓ Required provenance validated; network/model fallback disabled before deserialization.')

predictor = TabularPredictor.load(str(PREDICTOR_ROOT))
if predictor.problem_type not in {'binary', 'multiclass'}:
    raise RuntimeError(f'Expected a classification predictor, but loaded problem_type={predictor.problem_type!r}.')

FEATURE_COLUMNS = list(raw_features)
feature_metadata = getattr(predictor, 'feature_metadata_in', None)
if feature_metadata is None:
    raise RuntimeError('Loaded predictor exposes no input feature metadata for schema cross-check.')
loaded_feature_columns = list(feature_metadata.get_features())
if loaded_feature_columns != FEATURE_COLUMNS:
    raise RuntimeError('Loaded predictor feature schema does not match the validated pre-deserialization provenance schema.')

TARGET_COLUMN = run_metadata.get('target_column') or getattr(predictor, 'label', None)

summary = {
    'Artifact format': run_metadata['artifact_format'],
    'Artifact format version': run_metadata['artifact_format_version'],
    'AutoGluon runtime': AUTOGLUON_VERSION,
    'Problem type': predictor.problem_type,
    'Evaluation metric': str(predictor.eval_metric),
    'Target column': TARGET_COLUMN,
    'Required features': len(FEATURE_COLUMNS),
    'Models': ', '.join(predictor.model_names()),
    'Export mode': run_metadata.get('mode', 'not recorded'),
    'Base model': run_metadata.get('base_model', 'not recorded'),
    'Base revision': run_metadata.get('base_model_revision', 'not recorded'),
}
display(pd.Series(summary, name='Predictor').to_frame())

print('Required feature columns:')
display(pd.DataFrame({'feature': FEATURE_COLUMNS}))

if run_metadata:
    provenance_keys = [
        'model_source',
        'weights_sha256',
        'config_sha256',
        'data_source',
        'sample_revision',
        'train_rows_used',
        'holdout_rows',
        'independent_test_rows',
        'exported_at_utc',
    ]
    provenance = {k: run_metadata.get(k) for k in provenance_keys if run_metadata.get(k) is not None}
    if provenance:
        display(pd.Series(provenance, name='Recorded value').to_frame())


## 4. Upload new rows for inference

Upload one CSV containing the predictor's feature columns.

- Column order does not matter; the notebook reorders to the training order.
- Extra columns are preserved in the output but are not passed to the model.
- The target column is not required.
- Duplicate column names are rejected before pandas can rename them; missing required columns stop the run before inference.

The notebook uses only the feature schema recorded in the predictor provenance; it does not infer a new schema or fit preprocessing on the inference rows.


In [ ]:
import csv
import io


def read_inference_csv(payload):
    # Check original names before pandas can rename duplicate headers.
    text = payload.decode('utf-8-sig')
    rows = csv.reader(io.StringIO(text, newline=''))
    header = next((row for row in rows if row and not (len(row) == 1 and not row[0].strip())), [])
    seen = set()
    duplicates = []
    for name in header:
        if name in seen and name not in duplicates:
            duplicates.append(name)
        seen.add(name)
    if duplicates:
        raise ValueError(f'Inference CSV contains duplicate column names: {duplicates}')
    return pd.read_csv(io.BytesIO(payload))

uploaded = files.upload()
csvs = [(name, payload) for name, payload in uploaded.items() if name.lower().endswith('.csv')]
if len(csvs) != 1:
    raise RuntimeError('Upload exactly one inference CSV.')

csv_name, csv_payload = csvs[0]
new_data = read_inference_csv(csv_payload)

if new_data.columns.duplicated().any():
    duplicates = list(new_data.columns[new_data.columns.duplicated()])
    raise ValueError(f'Inference CSV contains duplicate column names: {duplicates}')

missing = [c for c in FEATURE_COLUMNS if c not in new_data.columns]
if missing:
    raise ValueError(f'Inference CSV is missing required feature columns: {missing}')

extra = [c for c in new_data.columns if c not in FEATURE_COLUMNS]
if extra:
    print(f'ℹ {len(extra)} extra column(s) will be preserved in predictions.csv but not used by the predictor: {extra}')

X = new_data.reindex(columns=FEATURE_COLUMNS).copy()
print(f'✓ Ready for inference: {len(X):,} rows × {len(FEATURE_COLUMNS)} required features.')
display(X.head())


## 5. Predict and download `predictions.csv`

The output preserves the uploaded columns and adds `prediction` plus one `probability_<class>` column per target class. If your CSV already has columns with those names, the cell stops rather than overwrite them.

**Reading the probabilities:** these are the predictor's per-class probabilities; calibration for your deployment population has not been established by this notebook. The `prediction` is the class with the highest returned probability (argmax). For a cost-sensitive decision, evaluate an application-specific threshold on held-out data rather than assuming argmax is optimal.

### When a check fails
| Error | Meaning | What to do |
|---|---|---|
| `Predictor ZIP checksum mismatch` | not the archive whose digest you were given | get the archive again; do not edit the expected digest |
| `Unsafe archive member path` / `Symlink entries are not allowed` | the archive tries to write outside its folder | reject it |
| `Expected exactly one AutoGluon predictor root` | wrong ZIP, or a ZIP of ZIPs | upload the `mitra-predictor.zip` from the export step |
| `Predictor was exported with AutoGluon X, but this runtime has Y` | version drift | install the recorded version |
| `Inference CSV is missing required feature columns` | schema mismatch | add the listed columns, with the training names |
| `Inference CSV contains duplicate column names` | ambiguous header | rename the duplicates |


In [ ]:
predictions = predictor.predict(X)
probabilities = predictor.predict_proba(X, as_multiclass=True)

reserved_output_columns = ['prediction'] + [f'probability_{label}' for label in probabilities.columns]
output_collisions = [name for name in reserved_output_columns if name in new_data.columns]
if output_collisions:
    raise ValueError(
        f'Inference CSV contains output column(s) reserved by this notebook: {output_collisions}. '
        'Rename or remove them before inference.'
    )

result = new_data.copy()
result['prediction'] = predictions.to_numpy()

for class_label in probabilities.columns:
    result[f'probability_{class_label}'] = probabilities[class_label].to_numpy()

OUTPUT_PATH = Path('/content/predictions.csv')
result.to_csv(OUTPUT_PATH, index=False)

display(result.head())
print(f'✓ Wrote {len(result):,} predictions to {OUTPUT_PATH}')
files.download(str(OUTPUT_PATH))


## What a successful artifact-inference run proves — and does not prove

A successful run proves that the supplied artifact passed the notebook's archive/provenance gates, loaded under the declared compatible runtime without network fallback, accepted the required feature schema, and produced class predictions plus per-class probabilities for the supplied rows.

It does **not** establish predictive quality, probability calibration, fairness, robustness to distribution shift, or suitability for a consequential deployment. Those claims require labelled evaluation data from the intended population and the relevant governance review.


## AI use and provenance

This inference tutorial was developed with substantial AI assistance under human direction and review.

- Original build: **GPT-5.6 Sol High** (OpenAI / ChatGPT), Agent Relay role: **Builder**
- Content revision: **Claude Fable 5.1** (Anthropic / Claude Code), Agent Relay role: **Reviewer and Builder**
- Review refinement & companion discoverability: **Gemini 3.8 Flash High** (Google DeepMind / Antigravity), Agent Relay role: **Builder**
- Base-model developer: **AutoGluon team, Amazon Web Services (AWS)**
- Predictor provenance: `tutorial_run_metadata.json` is required and validated before deserialization

AI attribution is **provenance, not sign-off** and does not independently verify correctness.
